In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("/home/ec2-user/studies/Haochen_food/code_micro/data")
COV_DIR = DATA_DIR / "covariates"


def baseline_rows(name):
    df = pd.read_csv(COV_DIR / name, low_memory=False)
    df["participant_id"] = df["participant_id"].astype(str).str.strip()
    df = df[df["research_stage"].astype(str).str.strip() == "00_00_visit"].copy()
    date_col = next(c for c in ["collection_date", "collection_timestamp", "research_stage_timestamp"] if c in df)
    df["_date"] = pd.to_datetime(df[date_col], errors="coerce", utc=True)
    return df.sort_values(["participant_id", "_date"]).drop_duplicates("participant_id")


def first_column(df, candidates):
    return next((c for c in candidates if c in df), None)


ids = pd.read_csv(DATA_DIR / "diet_experiment" / "diet_feature_matrix_baseline_full.csv", usecols=["participant_id"])
ids["participant_id"] = ids["participant_id"].astype(str).str.strip()

events = baseline_rows("events.csv")
age_col = first_column(events, ["age", "research_age", "age_at_visit"])
if age_col is None:
    visit = events["_date"]
    birth_month = pd.to_numeric(events["month_of_birth"], errors="coerce") if "month_of_birth" in events else 6
    events["age"] = visit.dt.year - pd.to_numeric(events["year_of_birth"], errors="coerce") + (visit.dt.month - birth_month) / 12
    age_col = "age"
events = events.rename(columns={age_col: "age", first_column(events, ["sex", "gender"]): "sex"})[["participant_id", "age", "sex"]]

anthro = baseline_rows("anthropometrics.csv")
if first_column(anthro, ["bmi", "BMI", "body_mass_index"]) is None:
    height = pd.to_numeric(anthro[next(c for c in anthro if "height" in c.lower())], errors="coerce")
    weight = pd.to_numeric(anthro[next(c for c in anthro if "weight" in c.lower())], errors="coerce")
    anthro["bmi"] = weight / np.where(height > 3, height / 100, height) ** 2
anthro = anthro.rename(columns={first_column(anthro, ["bmi", "BMI", "body_mass_index"]): "bmi"})[["participant_id", "bmi"]]

smoking = pd.read_csv(COV_DIR / "smoking_process_binary.csv", low_memory=False)
smoking["participant_id"] = smoking["participant_id"].astype(str).str.strip()
smoking["smoking_binary"] = pd.to_numeric(smoking["smoking_binary"], errors="coerce").where(lambda s: s.isin([0, 1]))
if "date_diff_days" in smoking:
    smoking["date_diff_days"] = pd.to_numeric(smoking["date_diff_days"], errors="coerce")
smoking = smoking.sort_values([c for c in ["participant_id", "date_diff_days"] if c in smoking])
smoking = smoking.drop_duplicates("participant_id")[["participant_id", "smoking_binary"]]

In [ ]:
oral = baseline_rows("oral_health.csv")
answer = {c: pd.to_numeric(oral[c], errors="coerce").where(lambda s: s >= 0) for c in
          ["oralhealth_bleeding_gums", "oralhealth_abscess_or_gingivitis", "oralhealth_teeth_mobility_lately"]}
bleeding, gingivitis, mobility = answer.values()
oral["oralhealth_soft_tissue_inflammation_level"] = np.select(
    [bleeding == 3, (bleeding == 2) | (gingivitis == 2)], [2, 1], default=0).astype(float)
oral.loc[bleeding.isna() & gingivitis.isna(), "oralhealth_soft_tissue_inflammation_level"] = np.nan
oral["oralhealth_periodontal_damage_level"] = np.select([mobility == 3, mobility == 2], [2, 1], default=0).astype(float)
oral.loc[mobility.isna(), "oralhealth_periodontal_damage_level"] = np.nan

covariates = (ids.merge(events, on="participant_id", how="left").merge(anthro, on="participant_id", how="left")
              .merge(smoking, on="participant_id", how="left")
              .merge(oral[["participant_id", "oralhealth_soft_tissue_inflammation_level",
                           "oralhealth_periodontal_damage_level"]], on="participant_id", how="left"))
covariates.to_csv(COV_DIR / "covariates_age_sex_bmi_smoking_oralhealth_baseline.csv", index=False)
print(covariates.drop(columns="participant_id").isna().sum())